1. Turn on **Internet** (needs to download
   `gensim.downloader` vectors, the `bert-base-uncased` weights, and `wordfreq` data).
2. Attach a Kaggle Dataset containing the framework
   (`game.py`,  `players/codemaster.py`, `players/guesser.py`,
   `players/cm_wordlist.txt`),

Saves result in
- `raw_log.jsonl`  one line per game (full result dict, appended as games finish)
- `results.csv`  table for analysis
- `anova_results.csv`  two-way ANOVA table (`total_turns ~ C(backend) * C(vocab_mode)`)

In [69]:
%pip install -q gensim wordfreq statsmodels sentence-transformers --no-warn-script-location

Note: you may need to restart the kernel to use updated packages.


In [70]:
import os, sys, json, time, shutil

ON_KAGGLE = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None
print("Running on Kaggle:" , ON_KAGGLE)

print(os.listdir("/kaggle/input/datasets/eyedeekey"))  # trovato la directory giusta
DATASET_DIR = "/kaggle/input/datasets/eyedeekey/codenames-gpt-python" 
FRAMEWORK_DIR = "/kaggle/working/codenames_framework"
if not os.path.exists(FRAMEWORK_DIR):
        shutil.copytree(DATASET_DIR, FRAMEWORK_DIR)
        print("copied")
else:
    FRAMEWORK_DIR = os.getcwd()
    print("check")

Running on Kaggle: True
['codenames-gpt-python']
check


In [71]:
BACKENDS      = ["word2vec", "glove", "bert"]   # nomic excluded 
VOCAB_MODES   = ["limited", "extended"]
N_SEEDS       = 2                              # how many test games
BASE_SEED     = 1
#20 test (120 totali) è troppo pesante, kaggle con CPU è crashato

#non ne sopporta neanche 5 (30 in totale)
#verificare se è dovuto alla mancanza di timeout oppure è proprio il server che non regge

#provare solo con il seed 0

EXTENDED_VOCAB_SIZE = 10000   # top-N most frequent English words (wordfreq) used as candidate pool
MAX_CANDIDATES_SCORED = 4000   # safety cap on how many candidates we score per clue (speed)
SIM_MARGIN = 0.05 #  questo faceva generare praticamente solo 1 come numero di clues
#SIM_MARGIN = 0.1   # how much better a word must be vs. the best rival color to add it to the clue count
MAX_CLUE_NUMBER = 4      # cap on how many words a single clue may target

RESULTS_DIR = "results_embedding"
os.makedirs(RESULTS_DIR, exist_ok=True)
RAW_LOG_PATH = os.path.join(RESULTS_DIR, "raw_log.jsonl")
RESULTS_CSV  = os.path.join(RESULTS_DIR, "results.csv")
ANOVA_CSV    = os.path.join(RESULTS_DIR, "anova_results.csv")

print(f"Grid size: {len(BACKENDS)} backends x {len(VOCAB_MODES)} vocab modes x {N_SEEDS} seeds "
      f"= {len(BACKENDS)*len(VOCAB_MODES)*N_SEEDS} games")


Grid size: 3 backends x 2 vocab modes x 2 seeds = 12 games


In [72]:
import random

#print(os.listdir("/kaggle/working/codenames_framework/players"))
#wordlist_path = os.path.join(FRAMEWORK_DIR, "players", "players","cm_wordlist.txt")
wordlist_path = os.path.join(FRAMEWORK_DIR, "codenames_framework", "players","cm_wordlist.txt")

#with open("/kaggle/working/codenames_framework/players/players/cm_wordlist.txt", "r") as f:
with open(wordlist_path, "r") as f:
    BOARD_WORD_POOL = [w.strip().upper() for w in f.read().splitlines() if w.strip()]

print(f"Board word pool size: {len(BOARD_WORD_POOL)}")

# "Limited" candidate vocabulary = exactly the game's own word pool (status quo today).
LIMITED_VOCAB = sorted(set(BOARD_WORD_POOL))

# "Extended" candidate vocabulary = top-N most frequent English words from wordfreq,
# unioned with the limited pool so nothing is lost, then capped for scoring speed.
from wordfreq import top_n_list
#Extended only take the top 8000 words for now

#TODO: "Slangs_and_Pop_culture" vocabulary
#TODO: FULL "Wikipedia_extended" vocabulary

freq_words = [w.upper() for w in top_n_list("en", EXTENDED_VOCAB_SIZE) if w.isalpha()]
EXTENDED_VOCAB = sorted(set(freq_words) | set(LIMITED_VOCAB))

print(f"Limited vocab size:  {len(LIMITED_VOCAB)}")
print(f"Extended vocab size: {len(EXTENDED_VOCAB)}")

VOCAB_BY_MODE = {"limited": LIMITED_VOCAB, "extended": EXTENDED_VOCAB}


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/codenames_framework/codenames_framework/players/cm_wordlist.txt'

In [73]:
import numpy as np

_BACKEND_CACHE = {}   # backend_name -> {"vectors": callable(word)->np.array or None, "has": callable(word)->bool}


def _l2_normalize(v):
    n = np.linalg.norm(v)
    return v / n if n > 0 else v


def cosine_sim(a, b):
    if a is None or b is None:
        return -1.0
    return float(np.dot(a, b))  # vectors are pre-normalized


def get_backend(name):
    '''Lazily load and cache an embedding backend. Returns dict with:
       - has(word) -> bool
       - vec(word) -> unit-normalized np.array or None
    '''
    if name in _BACKEND_CACHE:
        return _BACKEND_CACHE[name]

    if name == "word2vec":
        import gensim.downloader as gdl
        model = gdl.load("word2vec-google-news-300")

        def has(w):
            return w.lower() in model.key_to_index or w.capitalize() in model.key_to_index

        def vec(w):
            for cand in (w.lower(), w.capitalize(), w.upper()):
                if cand in model.key_to_index:
                    return _l2_normalize(model[cand])
            return None

    elif name == "glove":
        import gensim.downloader as gdl
        model = gdl.load("glove-wiki-gigaword-300")

        def has(w):
            return w.lower() in model.key_to_index

        def vec(w):
            wl = w.lower()
            if wl in model.key_to_index:
                return _l2_normalize(model[wl])
            return None

    elif name == "bert": #TODO: BERT locally instead of using HF token!
        import torch
        from transformers import AutoTokenizer, AutoModel

        tok = AutoTokenizer.from_pretrained("bert-base-uncased")
        bert_model = AutoModel.from_pretrained("bert-base-uncased")
        bert_model.eval()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        bert_model.to(device)

        _word_vec_cache = {}

        def _embed(w):
            if w in _word_vec_cache:
                return _word_vec_cache[w]
            with torch.no_grad():
                inputs = tok(w.lower(), return_tensors="pt").to(device)
                out = bert_model(**inputs).last_hidden_state[0]   # [seq_len, hidden]
                # mean-pool the sub-word tokens (excluding [CLS]/[SEP]) single-word,
                # out-of-sentence-context BERT embedding
                if out.shape[0] > 2:
                    v = out[1:-1].mean(dim=0)
                else:
                    v = out.mean(dim=0)
                v = v.cpu().numpy()
            v = _l2_normalize(v)
            _word_vec_cache[w] = v
            return v

        def has(w):
            return True  # BERT has no closed vocabulary, any word can be embedded via subwords

        def vec(w):
            return _embed(w)

    else:
        raise ValueError(f"Unknown backend: {name}")

    backend = {"has": has, "vec": vec}
    _BACKEND_CACHE[name] = backend
    return backend


In [74]:
#debug
import sys, os
print(FRAMEWORK_DIR in sys.path) 
#print(sys.path)

print("cwd:", os.getcwd())
print("FRAMEWORK_DIR:", FRAMEWORK_DIR)
print("in sys.path:", FRAMEWORK_DIR in sys.path)
print("players dir exists:", os.path.exists(os.path.join(FRAMEWORK_DIR, "players")))
#print("players contents:", os.listdir(os.path.join(FRAMEWORK_DIR, "players")))

#import players
#print("players module found at:", players.__file__)

for root, dirs, files in os.walk("/kaggle/working"):
    if "players" in dirs or "cm_wordlist.txt" in files:
        print("Found at:", root)

base = "/kaggle/working/codenames_framework"
print("Level 1 (framework root):", os.listdir(base))
print("Level 2 (players):", os.listdir(os.path.join(base, "players")))
print("Level 3 (players/players):", os.listdir(os.path.join(base, "players", "players"))) #found at lvl 3

True
cwd: /kaggle/working/codenames_framework
FRAMEWORK_DIR: /kaggle/working/codenames_framework
in sys.path: True
players dir exists: True
Found at: /kaggle/working/codenames_framework
Found at: /kaggle/working/codenames_framework/players
Level 1 (framework root): ['results', 'players', 'game.py', 'gpt_manager.py', 'env_setup.py', 'simple_example.py', 'run_game.py', 'results_embedding', '__pycache__', 'azure_manager.py']
Level 2 (players): ['cm_wordlist.txt', 'guesser_GPT.py', 'guesser.py', 'codemaster.py', 'codemaster_GPT.py', '__pycache__', 'codemaster_Local.py', 'guesser_Local.py', '__init__.py']


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/codenames_framework/players/players'

In [75]:
import shutil

outer = os.path.join(base, "players")
inner = os.path.join(base, "players", "players")

# move contents of inner up into outer, then remove empty inner folder
for item in os.listdir(inner):
    shutil.move(os.path.join(inner, item), os.path.join(outer, item)) #inner folder is now deleted

os.rmdir(inner)
print(os.listdir(outer)) 

#debug
print(os.path.exists(os.path.join(base, "players", "cm_wordlist.txt")))
print(os.path.exists(os.path.join(base, "players", "guesser.py")))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/codenames_framework/players/players'

In [77]:
init_path = os.path.join(base, "players", "__init__.py")
if not os.path.exists(init_path):
    open(init_path, "a").close()

if base not in sys.path:
    sys.path.insert(0, base)
os.chdir(base)

In [79]:
from players.codemaster import Codemaster
from players.guesser import Guesser


class EmbeddingCodemaster(Codemaster):
    '''Codemaster that scores a candidate-word vocabulary by cosine similarity
    to the team's remaining board words, minus the best-matching rival/assassin word.
    `backend` and `vocab_mode` are supplied via cmr_kwargs / cmb_kwargs at Game(...) construction,
    so no framework file needs to change to run this factorial experiment.'''

    def __init__(self, team="Red", backend="word2vec", vocab_mode="limited"):
        super().__init__()
        self.team = team
        self.backend_name = backend
        self.vocab_mode = vocab_mode
        self.backend = get_backend(backend)
        self.candidate_vocab = VOCAB_BY_MODE[vocab_mode]

    def set_game_state(self, words_on_board, key_grid):
        self.words = words_on_board
        self.key_grid = key_grid

    def _remaining_by_color(self):
        remaining = {"Red": [], "Blue": [], "Civilian": [], "Assassin": []}
        for w, k in zip(self.words, self.key_grid):
            if w[0] != "*":
                remaining[k].append(w)
        return remaining

    def get_clue(self):
        remaining = self._remaining_by_color()
        own_words = remaining[self.team]
        rival_color = "Blue" if self.team == "Red" else "Red"
        danger_words = remaining[rival_color] + remaining["Civilian"] + remaining["Assassin"]
        assassin_words = remaining["Assassin"]

        own_vecs = [(w, self.backend["vec"](w)) for w in own_words]
        own_vecs = [(w, v) for w, v in own_vecs if v is not None]
        danger_vecs = [self.backend["vec"](w) for w in danger_words if self.backend["vec"](w) is not None]
        assassin_vecs = [self.backend["vec"](w) for w in assassin_words if self.backend["vec"](w) is not None]

        board_words_set = set(self.words) | {w.replace("*", "") for w in self.words}
        candidates = self.candidate_vocab
        if len(candidates) > MAX_CANDIDATES_SCORED:
            candidates = random.sample(candidates, MAX_CANDIDATES_SCORED)

        best_clue, best_count, best_score = None, 1, -1e9
        for cand in candidates:
            if any(cand == bw or cand in bw or bw in cand for bw in self.words if bw[0] != "*"):
                continue
            cvec = self.backend["vec"](cand)
            if cvec is None or not own_vecs:
                continue

            sims_own = sorted(
                ((cosine_sim(cvec, v), w) for w, v in own_vecs), reverse=True
            )
            best_danger_sim = max((cosine_sim(cvec, v) for v in danger_vecs), default=-1.0)
            worst_assassin_sim = max((cosine_sim(cvec, v) for v in assassin_vecs), default=-1.0)
            if worst_assassin_sim > sims_own[0][0] - 0.02:
                continue  # too close to the assassin, skip this clue entirely

            count = 0
            for sim, _ in sims_own:
                if sim - best_danger_sim > SIM_MARGIN and count < MAX_CLUE_NUMBER:
                    count += 1
                else:
                    break
            if count == 0:
                continue

            score = sum(sim for sim, _ in sims_own[:count]) / count - best_danger_sim
            if score > best_score:
                best_clue, best_count, best_score = cand, count, score

        if best_clue is None:
            best_clue, best_count = random.choice(candidates), 1

        print(f"{self.team} Codemaster ({self.backend_name}/{self.vocab_mode}) clue: "
              f"({best_clue}, {best_count})")

        return [best_clue, best_count]


class EmbeddingGuesser(Guesser):
    '''Guesser that ranks remaining board words by cosine similarity to the clue
    and guesses in descending order, stopping when similarity drops sharply or the
    clue's guess budget (num + 1) is exhausted.'''

    def __init__(self, team="Red", backend="word2vec", vocab_mode="limited"):
        super().__init__()
        self.team = team
        self.backend = get_backend(backend)
        self.guesses_made = 0
        self.ranked = []

    def set_board(self, words_on_board):
        self.words = words_on_board

    def set_clue(self, clue, num):
        self.clue = clue
        self.num = int(num)
        self.guesses_made = 0
        cvec = self.backend["vec"](clue)
        remaining = [w for w in self.words if w[0] != "*"]
        scored = [(cosine_sim(cvec, self.backend["vec"](w)), w) for w in remaining]
        self.ranked = sorted(scored, reverse=True)
        self._rank_idx = 0

    def get_answer(self):
        remaining = [w for w in self.words if w[0] != "*"]
        # re-filter ranked list in case the board changed since set_clue
        self.ranked = [(s, w) for s, w in self.ranked if w in remaining]
        if not self.ranked:
            return None
        _, guess = self.ranked[self._rank_idx] if self._rank_idx < len(self.ranked) else self.ranked[-1]
        self.guesses_made += 1
        print(f"{self.team} Guesser guess:", guess)
        return guess

    def keep_guessing(self):
        self._rank_idx += 1
        if self.guesses_made >= self.num + 1:
            return False
        if self._rank_idx >= len(self.ranked):
            return False
        top_sim = self.ranked[0][0]
        next_sim = self.ranked[self._rank_idx][0]
        return (top_sim - next_sim) < 0.15   # stop once similarity drops off a cliff


In [ ]:
from game import Game
from itertools import product

def run_experiment():
    combos = list(product(BACKENDS, VOCAB_MODES))
    total = len(combos) * N_SEEDS
    done = 0
    t0 = time.time()

    with open(RAW_LOG_PATH, "a") as raw_f:
        for backend, vocab_mode in combos:
            kwargs = {"backend": backend, "vocab_mode": vocab_mode}
            for i in range(N_SEEDS):
                seed = BASE_SEED + i
                game_name = f"{backend}_{vocab_mode}_seed{seed}"
                try:
                    g = Game(
                        EmbeddingCodemaster, EmbeddingGuesser,
                        EmbeddingCodemaster, EmbeddingGuesser,
                        seed=seed, do_print=True, do_log=False,
                        game_name=game_name,
                        cmr_kwargs=kwargs, gr_kwargs=kwargs,
                        cmb_kwargs=kwargs, gb_kwargs=kwargs,
                    )
                    g.run()
                    record = {
                        "game_name": game_name,
                        "backend": backend,
                        "vocab_mode": vocab_mode,
                        "seed": seed,
                        "total_turns": len(g.get_move_history()),
                        "winner": g.game_winner,
                        "time_s": g.game_end_time - g.game_start_time,
                    }
                except Exception as e:
                    record = {
                        "game_name": game_name, "backend": backend, "vocab_mode": vocab_mode,
                        "seed": seed, "error": str(e),
                    }
                raw_f.write(json.dumps(record) + "\n")
                raw_f.flush()
                done += 1
                if done % 5 == 0 or done == total:
                    elapsed = time.time() - t0
                    print(f"[{done}/{total}] {game_name} -> "
                          f"{'ERROR: ' + record['error'] if 'error' in record else record['winner']} "
                          f"({elapsed:.0f}s elapsed)")

run_experiment()


seed: 1
RED TEAM TURN





____________________________KEY____________________________

   Blue      Civilian    Civilian    Civilian    Civilian   

 Assassin       Red        Blue        Blue        Blue     

    Red         Red         Red      Civilian       Red     

   Blue      Civilian      Blue      Civilian       Red     

    Red        Blue         Red         Red        Blue     
___________________________________________________________


___________________________BOARD___________________________

  FIGURE      STADIUM      FENCE       PART        MOUSE    

   FLUTE       FIRE         SUB        STOCK       SLUG     

   BUCK         GAS        BEAR        MATCH       HEART    

   HOOD        TRIP        SCUBA     DINOSAUR      CLIFF    

    OIL        CHEST       STAFF       ROBIN       GRACE    
___________________________________________________________


Red Codemaster (word2vec/limited) clue: (POLICE, 1)



Red Guesser guess: FIRE



___________________________

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


seed: 1
RED TEAM TURN





____________________________KEY____________________________

   Blue      Civilian    Civilian    Civilian    Civilian   

 Assassin       Red        Blue        Blue        Blue     

    Red         Red         Red      Civilian       Red     

   Blue      Civilian      Blue      Civilian       Red     

    Red        Blue         Red         Red        Blue     
___________________________________________________________


___________________________BOARD___________________________

  FIGURE      STADIUM      FENCE       PART        MOUSE    

   FLUTE       FIRE         SUB        STOCK       SLUG     

   BUCK         GAS        BEAR        MATCH       HEART    

   HOOD        TRIP        SCUBA     DINOSAUR      CLIFF    

    OIL        CHEST       STAFF       ROBIN       GRACE    
___________________________________________________________


Red Codemaster (bert/limited) clue: (CAST, 1)



Red Guesser guess: STAFF



___________________________BOARD

In [81]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

rows = []
with open(RAW_LOG_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
n_errors = df["error"].notna().sum() if "error" in df.columns else 0
if n_errors:
    print(f"WARNING: {n_errors} run into an error, ex")
df_ok = df[df["error"].isna()].copy() if "error" in df.columns else df.copy()

df_ok.to_csv(RESULTS_CSV, index=False)
print(f"Saved {len(df_ok)} clean rows to {RESULTS_CSV}")
df_ok.head()

model = ols("total_turns ~ C(backend) * C(vocab_mode)", data=df_ok).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

anova_table.to_csv(ANOVA_CSV)
print(f"\\nSaved ANOVA table to {ANOVA_CSV}")

Saved 48 clean rows to results_embedding/results.csv
                           sum_sq    df          F        PR(>F)
C(backend)                651.125   2.0  28.954209  1.249083e-08
C(vocab_mode)              75.000   1.0   6.670196  1.337869e-02
C(backend):C(vocab_mode)   23.625   2.0   1.050556  3.587524e-01
Residual                  472.250  42.0        NaN           NaN
\nSaved ANOVA table to results_embedding/anova_results.csv
